# Analyze a pkp-lite fit

Compares the fitted simplified model against the detailed-model
results in a `results/<run>/` folder, and reports the fit error using
the same per-run normalization pkp-lite's own fitting objective.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

## Pick a results folder

In [ ]:
results_dir = Path("results/input_Pittsburgh_2026-10-09_02")  # <- edit this

coefficients = yaml.safe_load((results_dir / "coefficients.yml").read_text())
print(coefficients["model"])
for name, value in coefficients["coefficients"].items():
    print(f"  {name}: {value:.4g}")

## Detailed vs. fitted model, per run

In [ ]:
run_ids = sorted(
    int(m.group(1)) for f in results_dir.glob("run*.csv")
    if (m := re.fullmatch(r"run(\d+)\.csv", f.name))
)

fig, axs = plt.subplots(1, len(run_ids), figsize=(4 * len(run_ids), 3), squeeze=False)
for i, ax in zip(run_ids, axs[0]):
    detailed = pd.read_csv(results_dir / f"run{i}.csv")
    model = pd.read_csv(results_dir / f"run{i}_model.csv")
    ax.plot(detailed["t"], detailed["volatiles"], label="detailed")
    ax.plot(model["t"], model["volatiles"], "--", label="fitted model")
    ax.set_title(f"run{i}")
    ax.set_xlabel("t [s]")
axs[0][0].set_ylabel("volatiles")
axs[0][-1].legend()
fig.tight_layout()

## Fit error per run

Same normalization as `evaluate_error`/`_residuals` (see
`pkplite/evolution_fit.py` / `pkplite/least_squares_fit.py`): each
run's MSE is divided by that run's own `volatiles` range, so this
reproduces (up to the `final_error_weight` term) what was actually
minimized during fitting.

In [ ]:
errors = {}
for i in run_ids:
    detailed = pd.read_csv(results_dir / f"run{i}.csv")["volatiles"].to_numpy()
    model = pd.read_csv(results_dir / f"run{i}_model.csv")["volatiles"].to_numpy()
    y_range = detailed.max() - detailed.min()
    errors[i] = np.mean((detailed - model) ** 2) / y_range**2

for i, err in errors.items():
    print(f"run{i}: {err:.4e}")
print(f"average (what pkp-lite minimizes): {np.mean(list(errors.values())):.4e}")